# MedQuAD dataset exploration
- Dataset is in `csv` format and contains 4 columns: question, answer, source and focus_area
- The total number of records is 16 412

In [1]:
import pandas as pd
ROOT = "../"

# Load CSV
df = pd.read_csv(ROOT + "dataset/medquad.csv")

print(f"Number of rows: {len(df)}")

# Display overall structure
display(df.head(10))
print(df.dtypes)

# Display a few full examples
for i in range(3):
    print(f"Example {i + 1}:")
    print(f"Question: {df.iloc[i]["question"]}")
    print(f"Answer: {df.iloc[i]["answer"]}")
    print(f"Focus Area: {df.iloc[i]["focus_area"]}")
    print("-" * 100)

Number of rows: 16412


,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
5,What is (are) Glaucoma ?,The optic nerve is a bundle of more than 1 mil...,NIHSeniorHealth,Glaucoma
6,What is (are) Glaucoma ?,Open-angle glaucoma is the most common form of...,NIHSeniorHealth,Glaucoma
7,Who is at risk for Glaucoma? ?,Anyone can develop glaucoma. Some people are a...,NIHSeniorHealth,Glaucoma
8,How to prevent Glaucoma ?,"At this time, we do not know how to prevent gl...",NIHSeniorHealth,Glaucoma
9,What are the symptoms of Glaucoma ?,"At first, open-angle glaucoma has no symptoms....",NIHSeniorHealth,Glaucoma


question      str
answer        str
source        str
focus_area    str
dtype: object
Example 1:
Question: What is (are) Glaucoma ?
Answer: Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops  There are several different types of glaucoma. Most of these involve the drainage system within the eye. At the front of the eye there is a small space called the anterior chamber. A clear fluid flows through this chamber and bathes and nourishes the nearby tissues. (Watch the video to learn more about glaucoma. To enlarge the video, click the brackets in the lower right-hand corner. To reduce the video, press the Escape (Esc) button on your keyboard.) In glaucoma, for still unknown reasons, the fluid drains too slowly out of the eye. As the fluid builds up, the pressure inside the eye rises. Unless this pressure is controlled, it may caus

# Filtering the dataset
- Our primary goal is getting the diesease name and its corresponding symptopms. In order to achieve that, we will filter the data to only get the records that either contain any of the specified `keywords` related to symptoms in either the `question` or the `answer` field
- After filtering the data by this logic, we get 14 884 records. There are 4985 unique `focus_areas` labels, and by inspecting a few of them, we can see that they indeed correspond to disease names.

In [2]:
import pandas as pd

# Load CSV
df = pd.read_csv(ROOT + "dataset/medquad.csv")

# We only need the data that contains some information about the disease and its symptoms
keywords = [
    "symptom",
    "signs",
    "cause",
    "disease",
    "condition",
    "syndrome"
]

# Create regex pattern
pattern = "|".join(keywords)

# Filter the data by checking if any of the keywords is either in the question or the answer field
filtered = df[
    df.iloc[:, 0].str.contains(pattern, case=False, na=False) |
    df.iloc[:, 1].str.contains(pattern, case=False, na=False)
]

# Show some filtered results
for i in range(5):
    print(f"Example {i}:")
    print(f"Question: {filtered.iloc[i]["question"]}")
    print(f"Answer: {filtered.iloc[i]["answer"]}")
    print(f"Focus Area: {filtered.iloc[i]["focus_area"]}")
    print("-" * 100)


print(f"Number of records: {filtered.shape[0]}")
print(f"Number of unique focus_areas: {filtered['focus_area'].nunique()}") # This will ideally contain disease names

# Show some unique focus areas
print(f"Example of focus_areas: {filtered["focus_area"].dropna().unique()[:25]}")

# Save the filtered data
#filtered.to_csv(ROOT + "dataset/medquad_filtered.csv")



Example 0:
Question: What is (are) Glaucoma ?
Answer: Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops  There are several different types of glaucoma. Most of these involve the drainage system within the eye. At the front of the eye there is a small space called the anterior chamber. A clear fluid flows through this chamber and bathes and nourishes the nearby tissues. (Watch the video to learn more about glaucoma. To enlarge the video, click the brackets in the lower right-hand corner. To reduce the video, press the Escape (Esc) button on your keyboard.) In glaucoma, for still unknown reasons, the fluid drains too slowly out of the eye. As the fluid builds up, the pressure inside the eye rises. Unless this pressure is controlled, it may cause damage to the optic nerve and other parts of the eye and result in loss of vision. O

# Dataset cleanup
- Even though the filtered data contain symptoms and the disease name (focus_area), the data is not clean. We cleanup the data by transforming the text to lowercase, removing leading and trailing spaces, removing URLs and cros references.

# Dataset cleanup
- Even though the filtered data contain symptoms and the disease name (focus_area), the data is not clean. We make the text lowercase, remove leading and trailing spaces, remove URLs and remove cross reference sentences
- We create column for the actual embeddeding

In [3]:
import re

def clean_text_rag(text):
    # If it's not string type
    if not isinstance(text, str):
        return ""

    text = text.lower() 
    text = re.sub(r"\s+", " ", text)  # Replace weird spacing with normal space
    
    # Remove cross reference sentences
    text = re.sub(r"(Read more|More information)[^.]*\.", "", text, flags=re.IGNORECASE)

    # Remove URLs
    text = re.sub(r"www\.\S+", "", text)
    
    return text.strip() # Remove leading and trailing white spaces

# Apply filters to each col
text_cols = filtered.select_dtypes(include="string").columns
for col in text_cols:
    filtered[col] = filtered[col].apply(clean_text_rag)

# Create newcolumn for embedding
filtered["text_for_rag"] = (
    "question: " + filtered["question"].fillna("") +
    " answer: " + filtered["answer"].fillna("")
)

display(filtered.head())

,question,answer,source,focus_area,text_for_rag
0,what is (are) glaucoma ?,glaucoma is a group of diseases that can damag...,nihseniorhealth,glaucoma,question: what is (are) glaucoma ? answer: gla...
1,what causes glaucoma ?,"nearly 2.7 million people have glaucoma, a lea...",nihseniorhealth,glaucoma,question: what causes glaucoma ? answer: nearl...
2,what are the symptoms of glaucoma ?,symptoms of glaucoma glaucoma can develop in o...,nihseniorhealth,glaucoma,question: what are the symptoms of glaucoma ? ...
3,what are the treatments for glaucoma ?,"although open-angle glaucoma cannot be cured, ...",nihseniorhealth,glaucoma,question: what are the treatments for glaucoma...
4,what is (are) glaucoma ?,glaucoma is a group of diseases that can damag...,nihseniorhealth,glaucoma,question: what is (are) glaucoma ? answer: gla...


# Create the embeddings

In [4]:
from sentence_transformers import SentenceTransformer

#model = SentenceTransformer("all-MiniLM-L6-v2")
model = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO")
embeddings = model.encode(filtered["text_for_rag"].tolist(), show_progress_bar=True)

filtered["embedding"] = list(embeddings)
display(filtered.head())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/466 [00:00<?, ?it/s]

,question,answer,source,focus_area,text_for_rag,embedding
0,what is (are) glaucoma ?,glaucoma is a group of diseases that can damag...,nihseniorhealth,glaucoma,question: what is (are) glaucoma ? answer: gla...,"[-0.39189, -0.37175816, -0.44920766, -0.717546..."
1,what causes glaucoma ?,"nearly 2.7 million people have glaucoma, a lea...",nihseniorhealth,glaucoma,question: what causes glaucoma ? answer: nearl...,"[-0.38811862, -0.65294784, -0.56563157, -0.618..."
2,what are the symptoms of glaucoma ?,symptoms of glaucoma glaucoma can develop in o...,nihseniorhealth,glaucoma,question: what are the symptoms of glaucoma ? ...,"[-0.35303274, -0.45500723, -0.2737506, -0.8178..."
3,what are the treatments for glaucoma ?,"although open-angle glaucoma cannot be cured, ...",nihseniorhealth,glaucoma,question: what are the treatments for glaucoma...,"[-0.47990558, -0.23519902, -0.37848306, -0.789..."
4,what is (are) glaucoma ?,glaucoma is a group of diseases that can damag...,nihseniorhealth,glaucoma,question: what is (are) glaucoma ? answer: gla...,"[-0.3206859, -0.3648812, -0.4305763, -0.684870..."


# Create RAG

In [5]:
import faiss
import numpy as np

embeddings = np.array(filtered["embedding"].tolist()).astype("float32")

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)


# Create query

In [7]:
def query_rag(prompt, k = 5):
    query_vec = model.encode([prompt]).astype("float32")
    D, I = index.search(query_vec, k=k)
    results = filtered.iloc[I[0]]
    return results

query_rag("Symptomps: Headache, feeling weak, sore throat, cough, really tired, runny nose")

,question,answer,source,focus_area,text_for_rag,embedding
8219,what are the symptoms of cough ?,"when you cough, mucus (a slimy substance) may ...",nhlbi,cough,question: what are the symptoms of cough ? ans...,"[-0.13024478, -0.48458353, -0.3863872, -0.4185..."
8325,what are the symptoms of pleurisy and other pl...,pleurisy the main symptom of pleurisy is a sha...,nhlbi,pleurisy and other pleural disorders,question: what are the symptoms of pleurisy an...,"[-0.20189041, -0.42842674, -0.3009936, -0.7417..."
910,what are the symptoms of small cell lung cancer ?,signs and symptoms of small cell lung cancer i...,cancergov,small cell lung cancer,question: what are the symptoms of small cell ...,"[0.018616175, -0.5351676, -0.20493786, -0.7092..."
1577,do you have information about cold and cough m...,"summary : sneezing, sore throat, a stuffy nose...",mplushealthtopics,cold and cough medicines,question: do you have information about cold a...,"[-0.3706403, -0.28104895, -0.35472324, -0.6082..."
309,what are the symptoms of copd ?,the most common symptoms of copd are a cough t...,nihseniorhealth,copd,question: what are the symptoms of copd ? answ...,"[-0.031966053, -0.52303106, -0.23881392, -0.42..."


## Create Best Matching 25 and query for it
- Cleaning special cleaning function that is pretty much the same as for the RAG, but on top of that includes removal of stop_words which should improve the performance
- We then build bm25

In [10]:
from rank_bm25 import BM25Okapi
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("punkt_tab")

stop_words = set(stopwords.words("english"))

def clean_text_BM25(text):
    # If it's not string type
    if not isinstance(text, str):
        return ""

    text = text.lower() 
    text = re.sub(r"\s+", " ", text)  # Replace weird spacing with normal space
    
    # Remove cross reference sentences
    text = re.sub(r"(Read more|More information)[^.]*\.", "", text, flags=re.IGNORECASE)

    # Remove URLs
    text = re.sub(r"www\.\S+", "", text)

    text = text.strip() # Remove leading and trailing white spaces

    # tokenize
    tokens = word_tokenize(text)

    # remove stopwords
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    return " ".join(tokens)


# Create newcolumn for embedding
filtered["text_for_bm25"] = (filtered["answer"].apply(clean_text_BM25))
display(filtered.head())


# Build BM25 index on the same answer text
tokenized_corpus = [doc.lower().split() if isinstance(doc, str) else [] for doc in filtered["text_for_bm25"].tolist()]
bm25 = BM25Okapi(tokenized_corpus)


[nltk_data] Downloading package punkt to /home/sagemaker-
[nltk_data]     user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/sagemaker-
[nltk_data]     user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/sagemaker-
[nltk_data]     user/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,question,answer,source,focus_area,text_for_rag,embedding,text_for_bm25
0,what is (are) glaucoma ?,glaucoma is a group of diseases that can damag...,nihseniorhealth,glaucoma,question: what is (are) glaucoma ? answer: gla...,"[-0.39189, -0.37175816, -0.44920766, -0.717546...",glaucoma group diseases damage eye 's optic ne...
1,what causes glaucoma ?,"nearly 2.7 million people have glaucoma, a lea...",nihseniorhealth,glaucoma,question: what causes glaucoma ? answer: nearl...,"[-0.38811862, -0.65294784, -0.56563157, -0.618...","nearly 2.7 million people glaucoma , leading c..."
2,what are the symptoms of glaucoma ?,symptoms of glaucoma glaucoma can develop in o...,nihseniorhealth,glaucoma,question: what are the symptoms of glaucoma ? ...,"[-0.35303274, -0.45500723, -0.2737506, -0.8178...",symptoms glaucoma glaucoma develop one eyes . ...
3,what are the treatments for glaucoma ?,"although open-angle glaucoma cannot be cured, ...",nihseniorhealth,glaucoma,question: what are the treatments for glaucoma...,"[-0.47990558, -0.23519902, -0.37848306, -0.789...","although open-angle glaucoma cured , usually c..."
4,what is (are) glaucoma ?,glaucoma is a group of diseases that can damag...,nihseniorhealth,glaucoma,question: what is (are) glaucoma ? answer: gla...,"[-0.3206859, -0.3648812, -0.4305763, -0.684870...",glaucoma group diseases damage eye 's optic ne...


# Query for BM25

In [11]:
def query_bm25(symptoms, k=5):
    # apply the same cleaning as corpus
    cleaned = clean_text_BM25(symptoms)
    tokens = cleaned.split()
    scores = bm25.get_scores(tokens)
    top_k = scores.argsort()[-k:][::-1]
    return filtered.iloc[top_k]

query_bm25("Headache, feeling weak, sore throat, cough, really tired, runny nose")

,question,answer,source,focus_area,text_for_rag,embedding,text_for_bm25
1577,do you have information about cold and cough m...,"summary : sneezing, sore throat, a stuffy nose...",mplushealthtopics,cold and cough medicines,question: do you have information about cold a...,"[-0.3706403, -0.28104895, -0.35472324, -0.6082...","summary : sneezing , sore throat , stuffy nose..."
8287,what are the symptoms of bronchitis ?,acute bronchitis acute bronchitis caused by an...,nhlbi,bronchitis,question: what are the symptoms of bronchitis ...,"[-0.16960326, -0.5738766, -0.3613927, -0.28572...",acute bronchitis acute bronchitis caused infec...
1602,what is (are) coronavirus infections ?,coronaviruses are common viruses that most peo...,mplushealthtopics,coronavirus infections,question: what is (are) coronavirus infections...,"[-0.36878505, -0.47844404, -0.3749417, -0.5285...",coronaviruses common viruses people get time l...
8219,what are the symptoms of cough ?,"when you cough, mucus (a slimy substance) may ...",nhlbi,cough,question: what are the symptoms of cough ? ans...,"[-0.13024478, -0.48458353, -0.3863872, -0.4185...","cough , mucus ( slimy substance ) may come . c..."
2336,what is (are) flu ?,flu is a respiratory infection caused by a num...,mplushealthtopics,flu,question: what is (are) flu ? answer: flu is a...,"[-0.43634576, -0.5218197, -0.4388858, -0.46935...",flu respiratory infection caused number viruse...


# Combine both queries

In [15]:
def combined_query(symptoms, k=5):
    res_rag = query_rag(symptoms, k)
    res_bm25 = query_bm25(symptoms, k)

    # Combine and drop duplicate rows
    combined = pd.concat([res_rag, res_bm25]).drop_duplicates(subset="question")
    return combined

combined_query("Headache, feeling weak, sore throat, cough, really tired, runny nose")

,question,answer,source,focus_area,text_for_rag,embedding,text_for_bm25
8219,what are the symptoms of cough ?,"when you cough, mucus (a slimy substance) may ...",nhlbi,cough,question: what are the symptoms of cough ? ans...,"[-0.13024478, -0.48458353, -0.3863872, -0.4185...","cough , mucus ( slimy substance ) may come . c..."
1577,do you have information about cold and cough m...,"summary : sneezing, sore throat, a stuffy nose...",mplushealthtopics,cold and cough medicines,question: do you have information about cold a...,"[-0.3706403, -0.28104895, -0.35472324, -0.6082...","summary : sneezing , sore throat , stuffy nose..."
1718,what is (are) sleep disorders ?,is it hard for you to fall asleep or stay asle...,mplushealthtopics,sleep disorders,question: what is (are) sleep disorders ? answ...,"[-0.3134205, -0.47546673, -0.16874737, -0.5248...",hard fall asleep stay asleep night ? wake feel...
8325,what are the symptoms of pleurisy and other pl...,pleurisy the main symptom of pleurisy is a sha...,nhlbi,pleurisy and other pleural disorders,question: what are the symptoms of pleurisy an...,"[-0.20189041, -0.42842674, -0.3009936, -0.7417...",pleurisy main symptom pleurisy sharp stabbing ...
2251,what is (are) sore throat ?,your throat is a tube that carries food to you...,mplushealthtopics,sore throat,question: what is (are) sore throat ? answer: ...,"[-0.25978607, -0.5434519, -0.39540285, -0.4960...",throat tube carries food esophagus air windpip...
8287,what are the symptoms of bronchitis ?,acute bronchitis acute bronchitis caused by an...,nhlbi,bronchitis,question: what are the symptoms of bronchitis ...,"[-0.16960326, -0.5738766, -0.3613927, -0.28572...",acute bronchitis acute bronchitis caused infec...
1602,what is (are) coronavirus infections ?,coronaviruses are common viruses that most peo...,mplushealthtopics,coronavirus infections,question: what is (are) coronavirus infections...,"[-0.36878505, -0.47844404, -0.3749417, -0.5285...",coronaviruses common viruses people get time l...
2336,what is (are) flu ?,flu is a respiratory infection caused by a num...,mplushealthtopics,flu,question: what is (are) flu ? answer: flu is a...,"[-0.43634576, -0.5218197, -0.4388858, -0.46935...",flu respiratory infection caused number viruse...
